# SNAP_Retailer_Location_data.csv

Executed source EDA. Only aggregate outputs are retained. Public inputs remain local. No causal inference or attendance model is used.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'public/data/manifest.json').exists())
manifest = json.loads((ROOT/'public/data/manifest.json').read_text())
areas = pd.read_json(ROOT/'public/data/areas.json')
counties = {c + ' County' for c in manifest['geography']}
print('Mode:', manifest['mode'], '| Version:', manifest['datasetVersion'])


In [ ]:
df = pd.read_csv(ROOT/'data/raw/SNAP_Retailer_Location_data.csv', dtype=str, encoding='utf-8-sig')
print('National/multiregion rows before filtering:', len(df))
df = df[(df.State == 'FL') & df.County.isin({c.removesuffix(' County').upper() for c in counties})].copy()
print('Known coordinate anomaly 546413:', int((df.Record_ID == '546413').sum()))
columns = ['Longitude', 'Latitude']
values = df[columns].apply(pd.to_numeric, errors='coerce')
print('Study-area rows:', len(df))
display(values.isna().sum().rename('missing'))
display(values.describe())
q1, q3 = values.quantile(.25), values.quantile(.75)
display(((values < q1 - 1.5*(q3-q1)) | (values > q3 + 1.5*(q3-q1))).sum().rename('IQR outliers; inspect, do not delete automatically'))
fig, ax = plt.subplots(figsize=(7,3))
values[columns[0]].dropna().hist(ax=ax,bins=20,color='#385981',grid=False)
ax.set(xlabel='Longitude (WGS84 longitude degrees)',ylabel='Retailer records',title='Study-area source distribution')
plt.show()


## Interpretation and limitations

The displayed row counts and missingness are observed from these local files. Numeric outliers are retained unless a documented source error exists. Statistical geography is the census tract; retailer locations are points, not service coverage. Estimates have mixed vintages and cannot establish individual need, travel time, attendance, or causal impact. ACS poverty/SNAP sentinel values remain missing. Tract-level associations do not describe individual households. The 2020 geometry joins by exact GEOID, never by tract label. See `docs/data_audit.md` and `docs/transformations.md` for exclusions and lineage.